In [1]:
# Installing spatial packages not included in the default Python environment.
# geopandas: reading/writing shapefiles and geographic data
# osmnx: downloading road networks from OpenStreetMap
# shapely: geometric operations (points inside polygons etc.)
!pip install geopandas osmnx requests shapely tqdm streetview datetime

In [5]:
# Imports and path definitions for this notebook.
# I define IMG_2012 and IMG_2019 as separate folders - one per time point.
# META_LOG is where I'll record the outcome of every download attempt
# (downloaded / no_coverage / failed) so I can audit what happened.
import geopandas as gpd
import osmnx as ox
import pandas as pd
import numpy as np
import requests
import os
import json
import time
from shapely.geometry import Point
from tqdm import tqdm
from streetview import search_panoramas
from datetime import datetime

#Defining my paths
base       = '.\\data' # update this to point to your local data directory
data_csv   = f'{base}\\Cleaned\\final_data.csv'
image_dir  = f'{base}\\Raw\\images_Chinese'
model_save = f'{base}\\Outputs\\Models\\best_model.pth'
LSOA_DIR     = f'{base}\\Raw'
IMG_2012     = f'{base}\\Raw\\Streetview_2012'  # Folder for 2012 images
IMG_2019     = f'{base}\\Raw\\Streetview_2019'  # Folder for 2019 images
META_LOG     = f'{base}\\Raw\\sv_metadata_log.csv'  #Downloading audit log
API_KEY      = 'YOUR_API_KEY_HERE'  # insert your own Google Street View Static API key

#Creating output folders if they dont exist already
os.makedirs(IMG_2012, exist_ok=True)
os.makedirs(IMG_2019, exist_ok=True)

In [3]:
# Loading the full England & Wales LSOA boundary shapefile downloaded from ONS.
# I use 2011 LSOA boundaries specifically because my IMD 2015 control variable
# and claimant count data both use 2011 LSOA codes - everything has to match.
lsoa_all = gpd.read_file(f'{LSOA_DIR}\\LSOA_2011_EW_BGC_V3.shp')
print("Total LSOAs in England & Wales:", len(lsoa_all))
print("Columns:", lsoa_all.columns.tolist())

Total LSOAs in England & Wales: 34753
Columns: ['LSOA11CD', 'LSOA11NM', 'LSOA11NMW', 'BNG_E', 'BNG_N', 'LONG', 'LAT', 'GlobalID', 'geometry']


In [4]:
# Filtering the national shapefile down to Greater Manchester only.
# GM consists of 10 metropolitan boroughs. I filter by checking whether
# the first word of the LSOA name matches any of these borough names.
# Then I reproject to WGS84 (epsg:4326) - this is the standard lat/lon
# coordinate system that the Google Street View API expects.
gm_districts = [
    'Bolton', 'Bury', 'Manchester', 'Oldham', 'Rochdale',
    'Salford', 'Stockport', 'Tameside', 'Trafford', 'Wigan'
]

# Filter: LSOA names start with any GM district name
gm_lsoas = lsoa_all[
    lsoa_all['LSOA11NM'].str.split().str[0].isin(gm_districts)
].copy().reset_index(drop=True)

print("GM LSOAs:", len(gm_lsoas)) #Should be 1673
print(gm_lsoas['LSOA11NM'].head(10)) #Checking first few

# Convert coordinate system to lat/lon
gm_lsoas = gm_lsoas.to_crs(epsg=4326) # Convert coordinate system to lat/lon

# Save
gm_lsoas.to_file(f'{LSOA_DIR}\\gm_lsoas.gpkg', driver='GPKG') # Save so I don't re-download
print("Saved.")

GM LSOAs: 1673
0    Bolton 005A
1    Bolton 005B
2    Bolton 001A
3    Bolton 003A
4    Bolton 003B
5    Bolton 003C
6    Bolton 005C
7    Bolton 003D
8    Bolton 005D
9    Bolton 014A
Name: LSOA11NM, dtype: str
Saved.


In [6]:
# Downloading the Greater Manchester road network from OpenStreetMap via osmnx.
# I then sample GPS points along roads within each LSOA — road-based sampling
# ensures my Street View requests land on actual streets with imagery coverage,
# not in parks or car parks where coverage is unlikely.
# POINTS_PER_LSOA = 6 gives 6 x 1,673 = 10,038 GPS points total.
# np.random.seed(42) makes the sampling deterministic and reproducible.
gm_lsoas = gpd.read_file(f'{LSOA_DIR}\\gm_lsoas.gpkg')

# Download road network for Greater Manchester once
print("Downloading GM road network (this takes 3-5 mins, once only)...")
G = ox.graph_from_place("Greater Manchester, England", network_type='drive')
nodes, edges = ox.graph_to_gdfs(G)
edges = edges.reset_index()
print("Road edges downloaded:", len(edges))

# Sample up to 8 points per LSOA along roads
POINTS_PER_LSOA = 6
np.random.seed(42)

sample_points = []

for _, lsoa in tqdm(gm_lsoas.iterrows(), total=len(gm_lsoas), desc="Sampling points"):
    lsoa_geom   = lsoa.geometry
    lsoa_code   = lsoa['LSOA11CD']
    
    # Finding all road segments that cross this LSOA's boundary
    edges_in = edges[edges.geometry.intersects(lsoa_geom)]
    
    if len(edges_in) == 0:
        continue #Skipping LSOAs with no roads (very rare)
    
    pts_found = []
    attempts  = 0
    
    while len(pts_found) < POINTS_PER_LSOA and attempts < 50:
        # Pick a random edge, sample a random point along it
        edge      = edges_in.sample(1).iloc[0] # Pick a random road segment
        line      = edge.geometry
        t         = np.random.uniform(0, 1)  # Random position along that segment (0=start, 1=end)
        pt        = line.interpolate(t, normalized=True) # Get the actual point at that position
        
        if lsoa_geom.contains(pt): # Only keep it if it falls inside this LSOA
            pts_found.append({
                'lsoa_code': lsoa_code,
                'lat':       pt.y,
                'lon':       pt.x
            })
        attempts += 1
    
    sample_points.extend(pts_found)

points_df = pd.DataFrame(sample_points)
points_df.to_csv(f'{base}\\Data\\Raw\\sample_points.csv', index=False)
print("Total sample points:", len(points_df))
print("Expected ~", len(gm_lsoas) * POINTS_PER_LSOA)

Road edges downloaded: 235162


Sampling points: 100%|██████████| 1673/1673 [01:00<00:00, 27.65it/s]

Total sample points: 10038
Expected ~ 10038


In [62]:
# Original approach used &date=YYYY-06 on the Static Street View
# API image endpoint. This parameter is NOT officially supported
# by that endpoint — it is silently ignored, meaning every request
# (regardless of requested year) returned the single most recent
# panorama available for that location. Verified via manual
# comparison of "2012" and "2019" images at matched coordinates,
# which were found to be visually identical.
#
# Fix: two-step lookup using search_panoramas() (unofficial but
# widely used endpoint) to retrieve ALL available panorama IDs and
# their actual capture dates for a location, then request the
# specific pano_id closest to the target date via the officially
# documented `pano=` parameter on the Static API.
#
# Tolerance: max_months_off=24 was tested against the original
# max_months_off=15; the observed maximum deviation across all
# downloaded images was 15.05 months in both configurations,
# indicating a genuine coverage ceiling in Greater Manchester
# Street View archives, not a rate-limiting or tolerance artifact.
# ============================================================

In [6]:
def find_best_pano(lat, lon, target_year, max_months_off=24):
    """
    Search all panoramas near (lat, lon), and return the one whose
    capture date is closest to June of target_year, within a tolerance.
    Returns None if nothing close enough exists.
    """
    try:
        panos = search_panoramas(lat=lat, lon=lon)
    except Exception as e:
        return None

    target = datetime(target_year, 6, 1)
    best_pano = None
    best_diff = None

    for p in panos:
        if not p.date:
            continue
        try:
            y, m = p.date.split('-')
            pdate = datetime(int(y), int(m), 1)
        except:
            continue

        diff_months = abs((pdate.year - target.year) * 12 + (pdate.month - target.month))

        if diff_months <= max_months_off:
            if best_diff is None or diff_months < best_diff:
                best_diff = diff_months
                best_pano = p

    if best_pano is None:
        return None
    return {'pano_id': best_pano.pano_id, 'actual_date': best_pano.date}


def download_pano_image(pano_id, save_path, api_key, retries=3):
    url = (
        f"https://maps.googleapis.com/maps/api/streetview"
        f"?size=640x640"
        f"&pano={pano_id}"
        f"&fov=90&heading=0&pitch=0"
        f"&key={api_key}"
    )
    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=15)
            if r.status_code == 200 and len(r.content) > 5000:
                with open(save_path, 'wb') as f:
                    f.write(r.content)
                return True
            return False
        except Exception as e:
            time.sleep(5)
    return False


In [ ]:
# ============================================================
# FULL RE-DOWNLOAD — replaces original (broken) date-based pipeline

# Skips any file already present (os.path.exists check) so this
# cell can be safely re-run without re-downloading existing images.
# ============================================================

In [7]:
############## DOWNLOAD LOOP CELLLLLLL ##############
points_df = pd.read_csv(f'{base}\\Data\\Raw\\sample_points.csv')
print("Points to process:", len(points_df))

log_rows = []

for year, image_dir in [(2012, IMG_2012), (2019, IMG_2019)]:
    print(f"\n--- Processing {year} ---")

    for _, row in tqdm(points_df.iterrows(), total=len(points_df), desc=str(year)):
        lsoa, lat, lon = row['lsoa_code'], row['lat'], row['lon']
        fname = f"{lsoa}_{lat:.6f}_{lon:.6f}_{year}.jpg"
        fpath = os.path.join(image_dir, fname)

        if os.path.exists(fpath):
            log_rows.append({'lsoa': lsoa, 'lat': lat, 'lon': lon,
                              'year': year, 'status': 'already_exists'})
            continue

        match = find_best_pano(lat, lon, year)

        if match is None:
            log_rows.append({'lsoa': lsoa, 'lat': lat, 'lon': lon,
                              'year': year, 'status': 'no_close_pano'})
            continue

        success = download_pano_image(match['pano_id'], fpath, API_KEY)

        log_rows.append({
            'lsoa': lsoa, 'lat': lat, 'lon': lon, 'year': year,
            'status': 'downloaded' if success else 'download_failed',
            'pano_id': match['pano_id'],
            'actual_date': match['actual_date']
        })

    pd.DataFrame(log_rows).to_csv(META_LOG, index=False)

print("Done. Log saved to", META_LOG)
print(pd.DataFrame(log_rows)['status'].value_counts())


Points to process: 10038

--- Processing 2012 ---


2012: 100%|██████████| 10038/10038 [1:24:34<00:00,  1.98it/s] 



--- Processing 2019 ---


2019: 100%|██████████| 10038/10038 [1:12:18<00:00,  2.31it/s]

Done. Log saved to E:\Warwick\EconDS\Dissertation\Data\Raw\sv_metadata_log.csv
status
downloaded         11279
no_close_pano       8792
download_failed        4
already_exists         1
Name: count, dtype: int64
